# Classical Machine Learning - subsampled #

In [1]:
from pathlib import Path
import time
import json
from tqdm.auto import tqdm
import gc

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.inspection import permutation_importance
import sys
from pathlib import Path

scripts_dir = (Path.cwd() / ".." / "scripts").resolve()
sys.path.append(str(scripts_dir))

from config import final_data_path

# Config
final_data_path = Path(final_data_path)
features_dir = final_data_path
results_dir = final_data_path / "modeling_results" / "rf"
results_dir.mkdir(parents=True, exist_ok=True)

layer_files = [
    "layer_a_bts",
    "layer_b_bts_meteostat",
    "layer_c_bts_gdelt",
    "layer_d_bts_meteostat_gdelt",
]
variants = ["", "_subsampled"]

target_col = "dep_del15"
non_feature_cols = ["flight_id", "year", target_col]

datasets = {}
for name in layer_files:
    for variant in variants:
        key = f"{name}{variant}"
        df = pd.read_parquet(features_dir / f"{key}_features.parquet")
        datasets[key] = df
        print(f"{key}: {df.shape}")


def get_train_test(key: str):
    """Split one dataset by year and drop non-feature columns from X."""
    df = datasets[key]
    train = df[df["year"] == 2024]
    test = df[df["year"] == 2025]

    feature_cols = [c for c in df.columns if c not in non_feature_cols]
    X_train, y_train = train[feature_cols], train[target_col]
    X_test, y_test = test[feature_cols], test[target_col]
    return X_train, y_train, X_test, y_test

/opt/anaconda3/envs/thesis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


layer_a_bts: (12028696, 28)
layer_a_bts_subsampled: (6966559, 28)
layer_b_bts_meteostat: (12028696, 32)
layer_b_bts_meteostat_subsampled: (6966559, 32)
layer_c_bts_gdelt: (12028696, 43)
layer_c_bts_gdelt_subsampled: (6966559, 43)
layer_d_bts_meteostat_gdelt: (12028696, 47)
layer_d_bts_meteostat_gdelt_subsampled: (6966559, 47)


Test best parameters on 1 subsampled layer.

In [2]:
X_train_full, y_train_full, X_test, y_test = get_train_test("layer_d_bts_meteostat_gdelt_subsampled")
print(f"Full subsample: {X_train_full.shape}")

search_sample_size = 100_000

X_train, _, y_train, _ = train_test_split(
    X_train_full, y_train_full,
    train_size=search_sample_size,
    stratify=y_train_full,
    random_state=42,
)
print(f"Search sample: {X_train.shape}")

base_params = {
    "max_features": "sqrt",
    "class_weight": "balanced",
    "oob_score": True,
    "n_jobs": 4,
    "random_state": 42,
}

param_grid = {
    "max_depth": [10, 20, 30, None],
    "min_samples_leaf": [1, 5, 10, 25],
    "n_estimators": [100, 200, 300, 500],
}

results_path = results_dir / "full_grid_tuning_trials.csv"

grid = list(ParameterGrid(param_grid))
print(f"Full grid: {len(grid)} combinations")

results = []
cumulative_time = 0.0

for combo in tqdm(grid, desc="full grid search"):
    params = {**base_params, **combo}

    start = time.time()
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    cumulative_time += elapsed

    result = {
        **combo,
        "oob_score": model.oob_score_,
        "fit_time_s": elapsed,
        "search_sample_size": search_sample_size,
    }
    results.append(result)

    del model
    gc.collect()

    # Persist immediately -- a crash on trial N doesn't lose trials 1..N-1
    pd.DataFrame([result]).to_csv(
        results_path, mode="a", header=not results_path.exists(), index=False
    )

    avg_time = cumulative_time / len(results)
    remaining = avg_time * (len(grid) - len(results))
    print(f"{combo} -> OOB={result['oob_score']:.4f}, "
          f"time={elapsed:.1f}s | est. remaining: {remaining/60:.1f} min")

tuning_df = pd.DataFrame(results)
tuning_df["oob_per_minute"] = tuning_df["oob_score"] / (tuning_df["fit_time_s"] / 60)
tuning_df = tuning_df.sort_values("oob_score", ascending=False)
tuning_df.to_csv(results_dir / "full_grid_tuning_summary.csv", index=False)

print(f"\nBest by OOB score:\n{tuning_df.iloc[0]}")
print(f"\nBest by efficiency (OOB/min):\n{tuning_df.sort_values('oob_per_minute', ascending=False).iloc[0]}")

tuning_df

Full subsample: (1000000, 44)
Search sample: (100000, 44)
Full grid: 64 combinations


full grid search:   0%|          | 0/64 [00:00<?, ?it/s]/opt/anaconda3/envs/thesis/lib/python3.12/site-packages/sklearn/ensemble/_forest.py:589: UserWarning: Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.
  warn(
full grid search:   2%|▏         | 1/64 [00:03<03:44,  3.56s/it]

{'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 100} -> OOB=0.6470, time=3.5s | est. remaining: 3.7 min


full grid search:   3%|▎         | 2/64 [00:10<05:47,  5.60s/it]

{'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 200} -> OOB=0.6482, time=7.0s | est. remaining: 5.4 min


...

full grid search:  98%|█████████▊| 63/64 [13:06<00:12, 12.34s/it]

{'max_depth': None, 'min_samples_leaf': 25, 'n_estimators': 300} -> OOB=0.6792, time=13.5s | est. remaining: 0.2 min


full grid search: 100%|██████████| 64/64 [13:29<00:00, 12.64s/it]

{'max_depth': None, 'min_samples_leaf': 25, 'n_estimators': 500} -> OOB=0.6798, time=22.8s | est. remaining: 0.0 min

Best by OOB score:
max_depth                       NaN
min_samples_leaf           1.000000
n_estimators             500.000000
oob_score                  0.775670
fit_time_s                28.679558
search_sample_size    100000.000000
oob_per_minute             1.622766
Name: 51, dtype: float64

Best by efficiency (OOB/min):
max_depth                 10.000000
min_samples_leaf           1.000000
n_estimators             100.000000
oob_score                  0.646990
fit_time_s                 3.517328
search_sample_size    100000.000000
oob_per_minute            11.036617
Name: 0, dtype: float64


,max_depth,min_samples_leaf,n_estimators,oob_score,fit_time_s,search_sample_size,oob_per_minute
51,NaN,1,500,0.77567,28.679558,100000,1.622766
34,30.0,1,300,0.77533,16.433319,100000,2.830822
35,30.0,1,500,0.77518,27.265014,100000,1.705878
50,NaN,1,300,0.77462,17.289292,100000,2.688207
49,NaN,1,200,0.77399,11.065696,100000,4.196699
...,...,...,...,...,...,...,...
15,10.0,25,500,0.64663,17.863588,100000,2.171893
14,10.0,25,300,0.64532,10.406573,100000,3.720648
13,10.0,25,200,0.64516,6.966048,100000,5.556895
4,10.0,5,100,0.64334,3.548509,100000,10.877921


Parameters for Classical ML models. Changeable section for parameter tuning. Beta added to weigh recall over precision (more costly to underestimate delays than overestimate).

In [3]:
rf_params = {
    "n_estimators": 300,
    "max_depth": 30,
    "min_samples_leaf": 10,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "oob_score": True,
    "n_jobs": 4,
    "random_state": 42,
}

beta = 1.2

full_layers = [
    "layer_a_bts",
    "layer_b_bts_meteostat",
    "layer_c_bts_gdelt",
    "layer_d_bts_meteostat_gdelt",
]

subsampled_layers = [
    "layer_a_bts_subsampled",
    "layer_b_bts_meteostat_subsampled",
    "layer_c_bts_gdelt_subsampled",
    "layer_d_bts_meteostat_gdelt_subsampled",
]

In [ ]:

def tune_threshold(y_true, oob_proba, beta=beta):
    """Pick the threshold that maximizes F-beta on OOB (in-sample, unbiased) predictions."""
    precision, recall, thresholds = precision_recall_curve(y_true, oob_proba)
    fbeta_scores = (
        (1 + beta**2) * precision * recall
        / (beta**2 * precision + recall + 1e-12)
    )
    best_idx = np.nanargmax(fbeta_scores[:-1])
    return thresholds[best_idx]

Run on all subsampled layers.

In [5]:
run_results = {}
layer_probas = {}
layer_importances = {}

for key in tqdm(subsampled_layers, desc="RF layers"):
    print(f"\n{'=' * 60}\n{key}\n{'=' * 60}")

    X_train, y_train, X_test, y_test = get_train_test(key)
    print(f"Train: {X_train.shape}, Test: {X_test.shape}")

    model = RandomForestClassifier(**rf_params)

    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    print(f"Fit time: {elapsed:.1f}s | OOB score: {model.oob_score_:.4f}")

    # OOB probabilities -> tune threshold without touching 2025 or refitting
    oob_proba = model.oob_decision_function_[:, 1]
    threshold = tune_threshold(y_train, oob_proba, beta=beta)
    print(f"Tuned threshold (F{beta}, from OOB): {threshold:.4f}")

    # Real evaluation on 2025 holdout, frozen threshold
    test_proba = model.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= threshold).astype(int)

    precision = precision_score(y_test, test_pred)
    recall = recall_score(y_test, test_pred)
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f}")

    metrics = {
        "f1": f1_score(y_test, test_pred),
        "precision": precision,
        "recall": recall,
        "auc_roc": roc_auc_score(y_test, test_proba),
        "auc_pr": average_precision_score(y_test, test_proba),
        "mcc": matthews_corrcoef(y_test, test_pred),
        "threshold": float(threshold),
        "oob_score": float(model.oob_score_),
        "fit_time_s": elapsed,
    }
    print(f"F1: {metrics['f1']:.4f} | AUC-ROC: {metrics['auc_roc']:.4f} | "
          f"AUC-PR: {metrics['auc_pr']:.4f} | MCC: {metrics['mcc']:.4f}")

    # Feature importance (impurity-based) — free byproduct of training, no
    # extra passes over the data
    importances = pd.Series(
        model.feature_importances_, index=X_train.columns
    ).sort_values(ascending=False)
    print(f"Top 10 features:\n{importances.head(10)}")

    layer_importances[key] = importances

    # Cache everything needed for future threshold experiments (e.g. a beta
    # sweep) without refitting the model
    layer_probas[key] = {
        "y_train": y_train.to_numpy(),
        "oob_proba": oob_proba,
        "y_test": y_test.to_numpy(),
        "test_proba": test_proba,
    }

    # Save probabilities (needed later for DeLong's test and threshold
    # experiments), metrics, and feature importances
    np.savez(
        results_dir / f"{key}_probs.npz",
        y_train=y_train.to_numpy(),
        oob_proba=oob_proba,
        y_test=y_test.to_numpy(),
        test_proba=test_proba,
    )
    with open(results_dir / f"{key}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    importances.to_csv(results_dir / f"{key}_importances.csv", header=["importance"])

    run_results[key] = metrics

# Combined summary table
summary_df = pd.DataFrame(run_results).T
summary_df.index.name = "layer"
summary_df.to_csv(results_dir / "rf_subsampled_summary.csv")
summary_df

RF layers:   0%|          | 0/4 [00:00<?, ?it/s]


layer_a_bts_subsampled
Train: (1000000, 25), Test: (5966559, 25)
Fit time: 122.1s | OOB score: 0.6856
Tuned threshold (F1.2, from OOB): 0.4464
Precision: 0.3131 | Recall: 0.6467


RF layers:  25%|██▌       | 1/4 [02:33<07:39, 153.07s/it]

F1: 0.4219 | AUC-ROC: 0.6731 | AUC-PR: 0.3565 | MCC: 0.2076
Top 10 features:
hour_sin            0.196322
origin_te           0.136599
dest_te             0.122979
distance            0.106827
crs_elapsed_time    0.105242
month_cos           0.071114
hour_cos            0.062169
month_sin           0.045298
dow_sin             0.045084
dow_cos             0.028963
dtype: float64

layer_b_bts_meteostat_subsampled
Train: (1000000, 29), Test: (5966559, 29)
Fit time: 141.4s | OOB score: 0.7083
Tuned threshold (F1.2, from OOB): 0.4538
Precision: 0.3311 | Recall: 0.6161


RF layers:  50%|█████     | 2/4 [05:35<05:41, 170.51s/it]

...

RF layers:  75%|███████▌  | 3/4 [08:58<03:05, 185.26s/it]

F1: 0.4237 | AUC-ROC: 0.6772 | AUC-PR: 0.3565 | MCC: 0.2149
Top 10 features:
hour_sin                    0.153284
origin_te                   0.063707
dest_te                     0.061159
crs_elapsed_time            0.050427
distance                    0.050119
month_cos                   0.048856
hour_cos                    0.044346
gdelt_total_count_lag7d     0.044230
gdelt_coerce_count_lag7d    0.043962
gdelt_total_count_lag3d     0.039651
dtype: float64

layer_d_bts_meteostat_gdelt_subsampled
Train: (1000000, 44), Test: (5966559, 44)
Fit time: 175.5s | OOB score: 0.7229
Tuned threshold (F1.2, from OOB): 0.4549
Precision: 0.3368 | Recall: 0.5933


RF layers: 100%|██████████| 4/4 [12:38<00:00, 189.66s/it]

F1: 0.4297 | AUC-ROC: 0.6860 | AUC-PR: 0.3675 | MCC: 0.2264
Top 10 features:
hour_sin            0.141213
temp_mean_12h       0.061173
origin_te           0.052679
dest_te             0.052042
wspd_mean_12h       0.047697
cldc_mean_12h       0.044077
distance            0.042753
crs_elapsed_time    0.042168
hour_cos            0.041174
month_cos           0.039395
dtype: float64


,f1,precision,recall,auc_roc,auc_pr,mcc,threshold,oob_score,fit_time_s
layer,,,,,,,,,
layer_a_bts_subsampled,0.421933,0.313103,0.646725,0.673140,0.356510,0.207643,0.446434,0.685587,122.101938
layer_b_bts_meteostat_subsampled,0.430714,0.331093,0.616085,0.685567,0.369668,0.225541,0.453810,0.708343,141.407222
layer_c_bts_gdelt_subsampled,0.423714,0.326103,0.604726,0.677162,0.356502,0.214926,0.453083,0.716778,162.527261
layer_d_bts_meteostat_gdelt_subsampled,0.429653,0.336769,0.593285,0.686012,0.367474,0.226426,0.454869,0.722872,175.505813


Beta sensitivity analysis: how does the tuned threshold change with different beta values?

In [6]:
betas_to_test = [1, 1.2, 1.5, 2]

sweep_results = []
for key, cached in layer_probas.items():
    for b in betas_to_test:
        threshold = tune_threshold(cached["y_train"], cached["oob_proba"], beta=b)
        pred = (cached["test_proba"] >= threshold).astype(int)

        sweep_results.append({
            "layer": key,
            "beta": b,
            "threshold": threshold,
            "precision": precision_score(cached["y_test"], pred),
            "recall": recall_score(cached["y_test"], pred),
            "f1": f1_score(cached["y_test"], pred),
            "mcc": matthews_corrcoef(cached["y_test"], pred),
        })

sweep_df = pd.DataFrame(sweep_results)
sweep_df.to_csv(results_dir / "beta_sweep.csv", index=False)
sweep_df

,layer,beta,threshold,precision,recall,f1,mcc
0,layer_a_bts_subsampled,1.0,0.497747,0.335896,0.540130,0.414205,0.209097
1,layer_a_bts_subsampled,1.2,0.446434,0.313103,0.646725,0.421933,0.207643
2,layer_a_bts_subsampled,1.5,0.391600,0.291133,0.748801,0.419259,0.199109
3,layer_a_bts_subsampled,2.0,0.309408,0.263260,0.870560,0.404269,0.174815
4,layer_b_bts_meteostat_subsampled,1.0,0.492095,0.351417,0.526472,0.421491,0.224394
5,layer_b_bts_meteostat_subsampled,1.2,0.453810,0.331093,0.616085,0.430714,0.225541
6,layer_b_bts_meteostat_subsampled,1.5,0.395704,0.302507,0.739887,0.429437,0.217817
7,layer_b_bts_meteostat_subsampled,2.0,0.311686,0.268108,0.874917,0.410441,0.189250
8,layer_c_bts_gdelt_subsampled,1.0,0.491221,0.346239,0.508338,0.411915,0.212453
9,layer_c_bts_gdelt_subsampled,1.2,0.453083,0.326103,0.604726,0.423714,0.214926
